# Notebook 04 — Hybrid Search + Reranking

### Why hybrid search?

❌ **Dense vectors fail on exact measurements**:  
They understand "brick wall" but cannot reliably match "4.5 inch", "1:4 ratio", "14 SWG". Numbers and specifications get lost in semantic embedding.

❌ **BM25 keywords fail on paraphrasing**:  
They match exact words but completely miss that "gypsum board ceiling" = "false ceiling work".

✅ **Hybrid search = best of both worlds**  
Run both methods, then combine the results using RRF.

**RRF = Reciprocal Rank Fusion**  
Each search method votes for results. RRF combines votes with a simple formula: `score = 1 / (rank + 60)`. Items that appear high in both lists get the highest total score.

In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from openai import OpenAI
from rank_bm25 import BM25Okapi
import pandas as pd
import numpy as np
from pathlib import Path
import json
import re
import time
from dotenv import load_dotenv

load_dotenv()

CHUNKS_PATH = Path("../data/processed/boq_chunks.csv")
COLLECTION_NAME = "boq_rates"

qdrant = QdrantClient(url="http://localhost:6333")
openai_client = OpenAI()

info = qdrant.get_collection(COLLECTION_NAME)
print(f"✅ Qdrant connected: {info.points_count} vectors")

df = pd.read_csv(CHUNKS_PATH)
print(f"✅ Chunks loaded: {df.shape}")
print(f"✅ Columns: {df.columns.tolist()}")

✅ Qdrant connected: 1346 vectors
✅ Chunks loaded: (1429, 13)
✅ Columns: ['chunk_id', 'embedding_text', 'description_short', 'description_full', 'section_title', 'work_category', 'rate', 'unit_norm', 'qty', 'source_file', 'sheet_name', 'rate_per_unit_label', 'est_tokens']


## Step 1: Build BM25 Index

BM25 = Best Match 25. It's the industry standard keyword ranking algorithm used in Elasticsearch, Solr, and every production search engine.

We build an in-memory BM25 index over all embedding_text fields.

In [2]:
def tokenize_boq(text: str) -> list[str]:
    """Tokenize BOQ text with special handling for measurements and ratios."""
    if pd.isna(text):
        return []
    text = str(text).lower()
    
    text = re.sub(r'([0-9]+\.?[0-9]*) *"', lambda m: m.group(1) + 'inch ', text)
    text = re.sub(r'([0-9]+\.?[0-9]*) *mm', lambda m: m.group(1) + 'mm ', text)
    text = re.sub(r'([0-9]+\.?[0-9]*) *tr\b', lambda m: m.group(1) + 'tr ', text)
    text = re.sub(r'([0-9]+) *swg\b', lambda m: m.group(1) + 'swg ', text)
    
    tokens = re.findall(r'[a-z0-9][a-z0-9\.\:\-\/]*', text)
    
    stopwords = {'and', 'or', 'the', 'of', 'in', 'to', 'for',
                 'with', 'as', 'per', 'at', 'on', 'is', 'are',
                 'be', 'by', 'an', 'a', 'all', 'etc', 'complete'}
    return [t for t in tokens if t not in stopwords and len(t) > 1]

corpus = df['embedding_text'].fillna('').tolist()
tokenized_corpus = [tokenize_boq(text) for text in corpus]
bm25 = BM25Okapi(tokenized_corpus)

print(f"✅ Built BM25 index over {len(corpus)} documents")
print()

test_query = "brick wall 4.5 inch cement sand"
test_tokens = tokenize_boq(test_query)
print(f"Test query: '{test_query}'")
print(f"Tokens: {test_tokens}")
print()

scores = bm25.get_scores(test_tokens)
top_indices = np.argsort(scores)[::-1][:5]

print("Top 5 BM25 results:")
print("-" * 60)
for i, idx in enumerate(top_indices):
    row = df.iloc[idx]
    print(f"{i+1:2d}. Score: {scores[idx]:.4f}")
    print(f"    Item: {row['description_short']}")
    print(f"    Rate: {row['rate_per_unit_label']}")
    print()

✅ Built BM25 index over 1429 documents

Test query: 'brick wall 4.5 inch cement sand'
Tokens: ['brick', 'wall', '4.5', 'inch', 'cement', 'sand']

Top 5 BM25 results:
------------------------------------------------------------
 1. Score: 12.6001
    Item: Providing and Making of the Brick Wall 9" Thick consisting of the First Class Burnt Brick with Ratio of 1:5 of approved Cement (D.G / Mapple Leaf / Lucky, Best Way) and Sand (Ravi) including cutting, wastage, hardware, labor for brick wall, Steel Bars to Tie, Mesh Jali to connect with R.C.C if needed, Leveling, Allignment, Scaffolding, Material Shifting, Freight & Cartage etc. Complete in all respects as per the instruction of the Architect / Client / Project Manager.
    Rate: Rs. 515 per sft

 2. Score: 12.5232
    Item: Providing and Making of the Brick Wall 4.5" Thick consisting of the First Class Burnt Brick with Ratio of 1:5 of approved Cement (D.G / Mapple Leaf / Lucky, Best Way) and Sand (Ravi) including cutting, wastage, hard

## BM25 vs Dense — Direct Comparison

Before building hybrid search, let's see where each method wins and where it fails.

In [3]:
def dense_search_only(query: str, top_k: int = 5) -> list[dict]:
    """Pure dense vector search via Qdrant."""
    vec = openai_client.embeddings.create(
        input=[query], 
        model="text-embedding-3-large",
        dimensions=3072
    ).data[0].embedding
    
    response = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=vec,
        limit=top_k,
        with_payload=True
    )
    
    return [{
        "chunk_id": r.payload["chunk_id"],
        "score": r.score, 
        "item": r.payload["description_short"],
        "rate": r.payload["rate_per_unit_label"],
        "file": r.payload["source_file"],
        "method": "dense"
    } for r in response.points]

def bm25_search_only(query: str, top_k: int = 5) -> list[dict]:
    """Pure BM25 keyword search."""
    tokens = tokenize_boq(query)
    scores = bm25.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        row = df.iloc[idx]
        results.append({
            "chunk_id": str(row['chunk_id']),
            "score": scores[idx],
            "item": str(row['description_short']),
            "rate": str(row['rate_per_unit_label']),
            "file": str(row['source_file']),
            "method": "bm25"
        })
    return results

def compare_methods(query: str):
    print(f"\n{'='*70}")
    print(f"QUERY: '{query}'")
    print(f"{'='*70}")
    
    dense = dense_search_only(query, 5)
    bm25_res = bm25_search_only(query, 5)
    
    print(f"\n{'DENSE SEMANTIC':35s} | {'BM25 KEYWORD':35s}")
    print(f"{'-'*35} | {'-'*35}")
    
    for d, b in zip(dense, bm25_res):
        d_text = f"{d['item'][:27]}"
        b_text = f"{b['item'][:27]}"
        print(f"{d_text:35s} | {b_text:35s}")

compare_methods("brick wall 4.5 inch 1:5 ratio")
compare_methods("gypsum board false ceiling 12mm")
compare_methods("UPVC pipe 32mm water supply")
compare_methods("14 SWG electrical wiring circuit")


QUERY: 'brick wall 4.5 inch 1:5 ratio'

DENSE SEMANTIC                      | BM25 KEYWORD                       
----------------------------------- | -----------------------------------
Brick Work 4.5" Thick               | Providing and Making of the        
Providing and Making of the         | Brick Work 4.5" Thick              
BRICKWORK 4 1/2"                    | Brick Work 9" Thick                
Providing and Making of the         | Providing and Making of the        
Providing and laying of any         | Providing and Making of the        

QUERY: 'gypsum board false ceiling 12mm'

DENSE SEMANTIC                      | BM25 KEYWORD                       
----------------------------------- | -----------------------------------
GYPSUM BOARD CEILING                | a) Gypsum board wall panell        
M.R. GYPSUM BOARD CEILING           | b) Gypsum board cladding ar        
Providing and Installing 12         | Providing & fixing in posit        
Providing & fixing in posit 

## Step 2: RRF Fusion

Reciprocal Rank Fusion is the current industry standard method for combining ranked lists.

Formula:  
```
rrf_score = 1 / (rank + 60)
```

60 is a standard smoothing constant.

In [4]:
def reciprocal_rank_fusion(ranked_lists: list[list[dict]], k: int = 60) -> list[dict]:
    """Combine multiple ranked result lists using Reciprocal Rank Fusion."""
    scores = {}
    payloads = {}
    
    for ranked_list in ranked_lists:
        for rank, item in enumerate(ranked_list):
            cid = item['chunk_id']
            rrf_score = 1.0 / (rank + k)
            scores[cid] = scores.get(cid, 0.0) + rrf_score
            payloads[cid] = item
    
    sorted_ids = sorted(scores.keys(), key=lambda x: scores[x], reverse=True)
    
    results = []
    for cid in sorted_ids:
        item = payloads[cid].copy()
        item['rrf_score'] = scores[cid]
        results.append(item)
    
    return results

list_A = [
    {"chunk_id": "AAA", "item": "Brick 4.5 inch", "rank": 1},
    {"chunk_id": "BBB", "item": "Brick 9 inch",   "rank": 2},
    {"chunk_id": "CCC", "item": "Plaster work",   "rank": 3}
]

list_B = [
    {"chunk_id": "BBB", "item": "Brick 9 inch",   "rank": 1},
    {"chunk_id": "AAA", "item": "Brick 4.5 inch", "rank": 2},
    {"chunk_id": "DDD", "item": "Marble floor",   "rank": 3}
]

fused = reciprocal_rank_fusion([list_A, list_B])

print("=== RRF FUSION DEMONSTRATION ===")
print(f"{'Chunk ID':8s} | {'Item':18s} | RRF Score")
print("-" * 55)
for r in fused:
    print(f"{r['chunk_id']:8s} | {r['item']:18s} | {r['rrf_score']:.6f}")
print("\n✅ Notice: BBB wins because it appears in both top 2 positions")

=== RRF FUSION DEMONSTRATION ===
Chunk ID | Item               | RRF Score
-------------------------------------------------------
AAA      | Brick 4.5 inch     | 0.033060
BBB      | Brick 9 inch       | 0.033060
CCC      | Plaster work       | 0.016129
DDD      | Marble floor       | 0.016129

✅ Notice: BBB wins because it appears in both top 2 positions


## Step 3: Full Hybrid Search Function

Complete pipeline flow:
```
Query → BM25 Search (top 20) ──┐
                               ├─ RRF Fusion → Top 10 → Cross Encoder → Top 5
Query → Dense Search (top 20) ─┘
```

Why `top_k_retrieve=20`? We cast a wide net so good results don't get missed.

In [5]:
def hybrid_search(
    query: str,
    top_k_retrieve: int = 20,
    top_k_fused: int = 10,
    work_category: str = None
) -> list[dict]:
    """Hybrid retrieval: BM25 + Dense Vector + RRF fusion."""
    
    query_vec = openai_client.embeddings.create(
        input=[query], model="text-embedding-3-large"
    ).data[0].embedding
    
    qdrant_filter = None
    if work_category:
        qdrant_filter = Filter(must=[
            FieldCondition(key="work_category", match=MatchValue(value=work_category))
        ])
    
    dense_hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vec,
        limit=top_k_retrieve,
        with_payload=True,
        query_filter=qdrant_filter
    )
    
    dense_results = []
    for rank, hit in enumerate(dense_hits.points):
        p = hit.payload
        dense_results.append({
            "chunk_id": p["chunk_id"],
            "description_short": p["description_short"],
            "description_full": p["description_full"],
            "work_category": p["work_category"],
            "rate": float(p["rate"]),
            "unit_norm": p["unit_norm"],
            "source_file": p["source_file"],
            "rate_per_unit_label": p["rate_per_unit_label"],
            "dense_score": float(hit.score),
            "dense_rank": rank
        })
    
    query_tokens = tokenize_boq(query)
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_top_idx = np.argsort(bm25_scores)[::-1][:top_k_retrieve]
    
    bm25_results = []
    for rank, idx in enumerate(bm25_top_idx):
        row = df.iloc[idx]
        if work_category and row["work_category"] != work_category:
            continue
        bm25_results.append({
            "chunk_id": str(row["chunk_id"]),
            "description_short": str(row["description_short"]),
            "description_full": str(row["description_full"]),
            "work_category": str(row["work_category"]),
            "rate": float(row["rate"]),
            "unit_norm": str(row["unit_norm"]),
            "source_file": str(row["source_file"]),
            "rate_per_unit_label": str(row["rate_per_unit_label"]),
            "bm25_score": float(bm25_scores[idx]),
            "bm25_rank": rank
        })
    
    fused = reciprocal_rank_fusion([dense_results, bm25_results])
    return fused[:top_k_fused]

print("Hybrid search test:")
results = hybrid_search("brick wall 4.5 inch 1:5 ratio", top_k_retrieve=20, top_k_fused=5)
for i, r in enumerate(results):
    print(f"  {i+1}. RRF:{r['rrf_score']:.5f} | {r['description_short'][:45]} | {r['rate_per_unit_label']}")

Hybrid search test:
  1. RRF:0.03306 | Brick Work 4.5" Thick | Rs. 315 per sft
  2. RRF:0.03254 | Providing and Making of the Brick Wall 9" Thi | Rs. 515 per sft
  3. RRF:0.03202 | Providing and Making of the Brick Wall 4.5" T | Rs. 320 per sft
  4. RRF:0.03128 | Brick Work 9" Thick | Rs. 485 per sft
  5. RRF:0.03080 | Providing and Making of the Brick Wall 9" Thi | Rs. 515 per sft


## Step 4: Cross-Encoder Reranking

Cross-encoder reads query + document together for more accurate ranking.

Model: `cross-encoder/ms-marco-MiniLM-L-6-v2`

In [6]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)
print("✓ Cross-encoder loaded")

def rerank_results(query: str, candidates: list[dict], top_k: int = 5) -> list[dict]:
    """Re-score candidates using cross-encoder."""
    if not candidates:
        return []
    
    pairs = [[query, str(c.get("description_full", c["description_short"]))[:512]] for c in candidates]
    scores = reranker.predict(pairs)
    
    for i, c in enumerate(candidates):
        c["reranker_score"] = float(scores[i])
    
    reranked = sorted(candidates, key=lambda x: x["reranker_score"], reverse=True)
    return reranked[:top_k]

candidates = hybrid_search("14 SWG electrical wiring circuit", top_k_retrieve=20, top_k_fused=10)
print("\nBEFORE reranking:")
for i, c in enumerate(candidates[:3]):
    print(f"  {i+1}. rrf={c['rrf_score']:.5f} | {c['description_short'][:50]}")

reranked = rerank_results("14 SWG electrical wiring circuit", candidates, top_k=3)
print("\nAFTER reranking:")
for i, r in enumerate(reranked):
    print(f"  {i+1}. score={r['reranker_score']:.3f} | {r['description_short'][:45]} | {r['rate_per_unit_label']}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Cross-encoder loaded

BEFORE reranking:
  1. rrf=0.03280 | Supply, installation  and commissioning of light c
  2. rrf=0.03037 | 3 Pin 15A Socket  (For Geyser)
  3. rrf=0.03016 | Supply,  installation  and  commissioning  of  lig

AFTER reranking:
  1. score=1.262 | 3 Pin 15A Socket  (For Geyser) | Rs. 12,500 per nos
  2. score=1.262 | Wiring and fixing of 2 TON floor standing ACs | Rs. 16,500 per nos
  3. score=1.262 | Hand Dryer (Hard wired) | Rs. 12,500 per nos


## Step 5: Complete query_boq() Pipeline

This is THE function the LangGraph agent will call.

In [7]:
def query_boq(query: str, top_k: int = 5, work_category: str = None, verbose: bool = True) -> list[dict]:
    """Complete BOQ rate lookup pipeline."""
    candidates = hybrid_search(query, top_k_retrieve=20, top_k_fused=10, work_category=work_category)
    
    if not candidates:
        if verbose:
            print(f"⚠ No candidates found for: '{query}'")
        return []
    
    results = rerank_results(query, candidates, top_k=top_k)
    
    if verbose:
        print(f"\n{'='*65}")
        print(f"QUERY: '{query}'")
        if work_category:
            print(f"FILTER: {work_category}")
        print(f"{'='*65}")
        for i, r in enumerate(results):
            print(f"\n[{i+1}] {r['description_short']}")
            print(f"     Rate: {r['rate_per_unit_label']}")
            print(f"     Source: {r['source_file']}")
            print(f"     Category: {r['work_category']}")
            print(f"     Scores: reranker={r['reranker_score']:.3f} rrf={r['rrf_score']:.5f}")
    
    return results

query_boq("brick wall 4.5 inch thick cement mortar 1:5")
query_boq("gypsum board false ceiling 12mm water resistant")
query_boq("light circuit wiring 2.5mm PVC conduit")


QUERY: 'brick wall 4.5 inch thick cement mortar 1:5'

[1] Providing and Making of the Brick Wall 4.5" Thick consisting of the First Class Burnt Brick with Ratio of 1:5 of approved Cement (D.G / Mapple Leaf / Lucky, Best Way) and Sand (Ravi) including cutting, wastage, hardware, labor for brick wall, Steel Bars to Tie, Mesh Jali to connect with R.C.C if needed, Leveling, Allignment, Scaffolding, Material Shifting, Freight & Cartage etc. Complete in all respects as per the instruction of the Architect / Client / Project Manager.
     Rate: Rs. 320 per sft
     Source: ID Works (G.F).xlsx
     Category: civil_id
     Scores: reranker=6.819 rrf=0.03252

[2] Brick Work 4.5" Thick
     Rate: Rs. 315 per sft
     Source: BOQ - 01 .xlsx
     Category: special_works
     Scores: reranker=4.600 rrf=0.03229

[3] Providing and laying of any thickness Bricks masonry wall up to any height, level  and floor using Solid with cement sand jointing mortar ratio 1:5 including leveling, alignment, scaffol

[{'chunk_id': 'e53811bf68eb7391ba98efc7ab890c56',
  'description_short': 'Supply, installation  and commissioning of light circuit\xa0wiring, from MCB in DB to Switch Board to be wired with 2x2.5mm sq. PVC insulated 450/750 V grade wire in and including cost of 25mm dia. heavy duty PVC conduit installed on roof slab, above false ceiling, or concealed in walls, or as required as per site conditions, all PVC conduit accessories, pull boxes, steel pull wires and 2.5 mm sq. PVC insulated wire of color green as circuit protective conductor (CPC), complete in all respects. Each circuit shall have independent CPC. Maximum wiring of 2 light circuits can be pulled through 25mm dia. PVC conduit.',
  'description_full': 'Circuit | Supply, installation  and commissioning of light circuit\xa0wiring, from MCB in DB to Switch Board to be wired with 2x2.5mm sq. PVC insulated 450/750 V grade wire in and including cost of 25mm dia. heavy duty PVC conduit installed on roof slab, above false ceiling, or c

## Category Filter Test

The work_category filter narrows search to one domain.

In [8]:
print("=== WITHOUT filter: 'wiring 2.5mm' ===")
query_boq("wiring 2.5mm circuit", work_category=None, top_k=2)

print("\n=== WITH electrical filter: 'wiring 2.5mm' ===")
query_boq("wiring 2.5mm circuit", work_category="electrical_elv", top_k=2)

print("\n=== WITH civil_id filter: 'tile floor' ===")
query_boq("tile floor installation", work_category="civil_id", top_k=2)

=== WITHOUT filter: 'wiring 2.5mm' ===

QUERY: 'wiring 2.5mm circuit'

[1] Supply, installation  and commissioning of light circuit wiring, from MCB in DB to Switch Board to be wired with 2x2.5mm sq. PVC insulated 450/750 V grade wire in and including cost of 25mm dia. heavy duty PVC conduit installed on roof slab, above false ceiling, or concealed in walls, or as required as per site conditions, all PVC conduit accessories, pull boxes, steel pull wires and 2.5 mm sq. PVC insulated wire of color green as circuit protective conductor (CPC), complete in all respects. Each circuit shall have independent CPC. Maximum wiring of 2 light circuits can be pulled through 25mm dia. PVC conduit.
     Rate: Rs. 19,700 per nos
     Source: Circuit Wiring.xlsx
     Category: electrical_elv
     Scores: reranker=4.554 rrf=0.03333

[2] Supply,  installation  and  commissioning  of  light  circuit wiring,  to  be  wired  with  3x2.5mm  sq.  (1P+1N+1CPC) PVC  insulated  300/500 V  grade  wire,  manufactu

[{'chunk_id': '863b8d0bbe33afcd2dc35d38b9542f06',
  'description_short': 'Porcelain Tile',
  'description_full': 'TILE INSTALLATION (Fixing Only) — Laying of owner provided tiles at walls and floor using Sika/Stile/ressichem tile bond and base Plaster in Ratio (1:5) 1½" - 2½" thick for floor, cement sand mortar using approved Cement and clean Local dust free sand, as per site requirement, unsanded grout, tile levelers, color pigments, spacers in required size, cutting, curing, cleaning, grooving, chipping if required for leveling and bonding, wastage, all fixing accessories and arrangements, etc. Complete in all respects and as per drawing, patterns and directions of the Architect. | Porcelain Tile',
  'work_category': 'civil_id',
  'rate': 250.0,
  'unit_norm': 'sft',
  'source_file': '1.Civil.xlsx',
  'rate_per_unit_label': 'Rs. 250 per sft',
  'bm25_score': 6.940510805859159,
  'bm25_rank': 5,
  'rrf_score': 0.03333333333333333,
  'reranker_score': 7.906664848327637},
 {'chunk_id': 

## Accuracy Evaluation on Known Items
We test the full pipeline on items we KNOW are in 
the knowledge base. For each query we check:
  - Does the correct item appear in top-3?
  - Is the rate in the expected range?
  - Is the unit correct?

Also note a key finding from testing:
"ms-marco cross-encoder is trained on web passages,
not technical BOQ specs. When reranker scores are 
all equal (like 1.262), it means the model couldn't 
distinguish — fall back to RRF order in that case."

In [9]:
known_items = [
    {
        "query": "brick work 4.5 inch thick wall cement mortar",
        "expected_rate_range": (300, 400),
        "expected_unit": "sft",
        "note": "core civil item"
    },
    {
        "query": "gypsum board ceiling 12mm false ceiling",
        "expected_rate_range": (350, 550),
        "expected_unit": "sft",
        "note": "most common ceiling type"
    },
    {
        "query": "cement plaster internal walls 12mm smooth finish",
        "expected_rate_range": (50, 250),
        "expected_unit": "sft",
        "note": "basic finishing work"
    },
    {
        "query": "split AC 4 ton cooling capacity installation",
        "expected_rate_range": (15000, 25000),
        "expected_unit": "nos",
        "note": "HVAC — SAC units"
    },
    {
        "query": "light circuit wiring PVC conduit 2.5mm",
        "expected_rate_range": (8000, 25000),
        "expected_unit": "nos",
        "note": "electrical circuits"
    },
    {
        "query": "UPVC pipe 32mm diameter water supply",
        "expected_rate_range": (200, 800),
        "expected_unit": "rft",
        "note": "plumbing — weak category"
    },
    {
        "query": "marble flooring installation dry bond adhesive",
        "expected_rate_range": (500, 2500),
        "expected_unit": "sft",
        "note": "stone flooring"
    },
    {
        "query": "waterproofing chemical pudlo cement plaster walls",
        "expected_rate_range": (25, 500),
        "expected_unit": "sft",
        "note": "waterproofing treatment"
    },
    {
        "query": "CCTV camera IP based POE outdoor installation",
        "expected_rate_range": (2000, 10000),
        "expected_unit": "nos",
        "note": "ELV systems"
    },
    {
        "query": "fire alarm control panel FACP addressable intelligent",
        "expected_rate_range": (300000, 700000),
        "expected_unit": "nos",
        "note": "fire fighting — rare items"
    }
]

passed = 0
failed = 0
fail_details = []

print(f"{'#':2} {'QUERY':38s} {'STATUS':8s} "
      f"{'TOP RESULT':25s} {'RATE FOUND':18s} {'EXPECTED RANGE'}")
print("-" * 110)

for idx, item in enumerate(known_items):
    results = query_boq(item["query"], top_k=3, verbose=False)
    
    found      = False
    top_label  = "—"
    found_rate = "—"
    
    for r in results:
        rate = r["rate"]
        unit = r["unit_norm"]
        lo, hi = item["expected_rate_range"]
        
        if lo <= rate <= hi and unit == item["expected_unit"]:
            found      = True
            top_label  = r["description_short"][:22]
            found_rate = r["rate_per_unit_label"]
            break
    
    if found:
        passed += 1
        status = "✅ PASS"
    else:
        failed += 1
        status = "❌ FAIL"
        if results:
            top_label  = results[0]["description_short"][:22]
            found_rate = results[0]["rate_per_unit_label"]
        fail_details.append({
            "query": item["query"],
            "note":  item["note"],
            "got":   found_rate
        })
    
    q_short = item["query"][:36]
    lo, hi  = item["expected_rate_range"]
    exp_str = f"Rs.{lo:,}–{hi:,}/{item['expected_unit']}"
    print(f"{idx+1:2} {q_short:38s} {status:8s} "
          f"{top_label:25s} {found_rate:18s} {exp_str}")

print()
print(f"{'='*60}")
print(f"RESULT: {passed}/10 passed  ({passed*10}% accuracy)")
print(f"Failed: {failed}/10")
if fail_details:
    print(f"\nFailed queries detail:")
    for f in fail_details:
        print(f"  • {f['note']}: got {f['got']}")
print(f"{'='*60}")

#  QUERY                                  STATUS   TOP RESULT                RATE FOUND         EXPECTED RANGE
--------------------------------------------------------------------------------------------------------------
 1 brick work 4.5 inch thick wall cemen   ✅ PASS   Brick Work 4.5" Thick     Rs. 315 per sft    Rs.300–400/sft
 2 gypsum board ceiling 12mm false ceil   ✅ PASS   M.R. GYPSUM BOARD CEIL    Rs. 485 per sft    Rs.350–550/sft
 3 cement plaster internal walls 12mm s   ✅ PASS   Providing and applying    Rs. 90 per sft     Rs.50–250/sft
 4 split AC 4 ton cooling capacity inst   ✅ PASS   SAC-1 to 11:
Nominal C    Rs. 17,000 per nos Rs.15,000–25,000/nos
 5 light circuit wiring PVC conduit 2.5   ✅ PASS   Supply, installation      Rs. 19,700 per nos Rs.8,000–25,000/nos
 6 UPVC pipe 32mm diameter water supply   ✅ PASS   Providing and Fixing o    Rs. 310 per rft    Rs.200–800/rft
 7 marble flooring installation dry bon   ✅ PASS   VANITY MARBLE  (Base R    Rs. 1,310 per sft  Rs.500

## Key Finding: Reranker Score Threshold
During testing we observed that ms-marco cross-encoder
sometimes assigns equal scores (e.g. 1.262) to multiple
BOQ items. This means the model couldn't differentiate.

Fix: in the agent, we will use this logic:
  - If top reranker score > 3.0 → trust reranker order
  - If top reranker score <= 3.0 → use RRF order instead

This gives us the best of both worlds:
  - High confidence queries → cross-encoder precision
  - Low confidence queries → RRF stability

In [10]:
def smart_rank(
    query: str,
    top_k: int = 5,
    work_category: str = None,
    reranker_threshold: float = 3.0
) -> list[dict]:
    """
    Smart ranking with reranker threshold fallback.
    
    If cross-encoder is confident (top score > threshold),
    use reranker order. Otherwise fall back to RRF order.
    This handles cases where ms-marco model gives equal
    scores to all candidates (can't distinguish BOQ specs).
    
    Returns list of result dicts with 'final_rank' added.
    """
    # Hybrid retrieval
    candidates = hybrid_search(
        query,
        top_k_retrieve=20,
        top_k_fused=10,
        work_category=work_category
    )
    if not candidates:
        return []
    
    # Rerank
    reranked = rerank_results(query, candidates, top_k=top_k)
    
    if not reranked:
        return []
    
    top_score = reranked[0]["reranker_score"]
    
    if top_score >= reranker_threshold:
        # Reranker confident — use reranker order
        order   = "reranker"
        results = reranked
    else:
        # Reranker not confident — use RRF order
        order   = "rrf_fallback"
        results = sorted(candidates,
                         key=lambda x: x["rrf_score"],
                         reverse=True)[:top_k]
        # Still attach reranker scores for transparency
        for r in results:
            if "reranker_score" not in r:
                r["reranker_score"] = 0.0
    
    # Add metadata
    for i, r in enumerate(results):
        r["final_rank"]    = i + 1
        r["ranking_method"] = order
    
    return results

# Test smart_rank on the two problem queries
print("=== SMART RANK TEST ===\n")

print("Query: '14 SWG electrical wiring'")
print("(reranker was confused here — should fall back to RRF)")
r1 = smart_rank("14 SWG electrical wiring circuit", top_k=3)
for r in r1:
    print(f"  [{r['final_rank']}] method={r['ranking_method']} | "
          f"reranker={r['reranker_score']:.3f} | "
          f"{r['description_short'][:45]} | "
          f"{r['rate_per_unit_label']}")

print()
print("Query: 'gypsum board false ceiling 12mm'")
print("(reranker was confident here — should use reranker order)")
r2 = smart_rank("gypsum board false ceiling 12mm water resistant",
                top_k=3)
for r in r2:
    print(f"  [{r['final_rank']}] method={r['ranking_method']} | "
          f"reranker={r['reranker_score']:.3f} | "
          f"{r['description_short'][:45]} | "
          f"{r['rate_per_unit_label']}")

=== SMART RANK TEST ===

Query: '14 SWG electrical wiring'
(reranker was confused here — should fall back to RRF)
  [1] method=rrf_fallback | reranker=-5.694 | Supply, installation  and commissioning of li | Rs. 19,700 per nos
  [2] method=rrf_fallback | reranker=1.262 | 3 Pin 15A Socket  (For Geyser) | Rs. 12,500 per nos
  [3] method=rrf_fallback | reranker=-4.240 | Supply,  installation  and  commissioning  of | Rs. 9,300 per nos

Query: 'gypsum board false ceiling 12mm'
(reranker was confident here — should use reranker order)
  [1] method=reranker | reranker=8.798 | Providing and Installing 12 mm thick  Gypsum  | Rs. 445 per sft
  [2] method=reranker | reranker=6.218 | M.R. GYPSUM BOARD CEILING | Rs. 485 per sft
  [3] method=reranker | reranker=6.218 | DOUBLE HT. CEILING | Rs. 490 per sft


Update ../src/rateiq/hybrid_search.py to add
smart_rank() as a method of BOQSearcher class.

Add this method to the BOQSearcher class
(append after the existing search() method):

    def smart_search(
        self,
        query: str,
        top_k: int = 5,
        work_category: str = None,
        reranker_threshold: float = 3.0
    ) -> list[dict]:
        """
        Search with reranker threshold fallback.
        Uses reranker order if confident (score > threshold),
        else falls back to RRF order.
        This is the method the LangGraph agent calls.
        """
        candidates = self._hybrid(query, 20, 10, work_category)
        if not candidates:
            return []
        
        reranked = self._rerank(query, candidates, top_k)
        if not reranked:
            return []
        
        top_score = reranked[0]["reranker_score"]
        
        if top_score >= reranker_threshold:
            results = reranked
            order   = "reranker"
        else:
            results = sorted(candidates,
                             key=lambda x: x["rrf_score"],
                             reverse=True)[:top_k]
            for r in results:
                r.setdefault("reranker_score", 0.0)
            order = "rrf_fallback"
        
        for i, r in enumerate(results):
            r["final_rank"]     = i + 1
            r["ranking_method"] = order
        
        return results

Write the COMPLETE updated hybrid_search.py to 
../src/rateiq/hybrid_search.py with:
  - tokenize_boq()
  - reciprocal_rank_fusion()
  - BOQSearcher class with methods:
      __init__()         → load df, build bm25, load reranker
      _embed()           → embed a query text
      _hybrid()          → BM25 + dense + RRF
      _rerank()          → cross-encoder rerank
      search()           → full pipeline (original)
      smart_search()     → pipeline with threshold fallback

After writing the file print:
"✓ Updated hybrid_search.py saved"

Then verify by re-importing:
  import importlib
  import sys
  # Remove cached module
  for key in list(sys.modules.keys()):
      if 'rateiq' in key:
          del sys.modules[key]
  from rateiq.hybrid_search import BOQSearcher
  s = BOQSearcher()
  test = s.smart_search("brick wall 4.5 inch", top_k=2)
  print(f"Import test: {test[0]['description_short'][:40]}")
  print(f"Method used: {test[0]['ranking_method']}")
  print("✓ Module working correctly")

## Notebook 04 Complete — What We Built

| Component          | Details                              |
|--------------------|--------------------------------------|
| BM25 Index         | 1429 docs, tokenize_boq() preserves measurements |
| Dense Search       | Qdrant, text-embedding-3-large, 3072 dims |
| RRF Fusion         | k=60, top-20 from each → top-10 fused |
| Cross-Encoder      | ms-marco-MiniLM-L-6-v2, threshold=3.0 |
| smart_search()     | Auto-selects reranker vs RRF order   |
| Accuracy           | X/10 known items (from Cell 17)      |

## Key Insight — Reranker Limitation
The ms-marco cross-encoder works well for natural 
language queries but struggles with pure technical 
specs ("14 SWG", pipe sizes). The threshold fallback 
ensures we always return sensible results.

## What Goes Into the Agent (Notebook 06)
The LangGraph agent will call:
  searcher = BOQSearcher()
  results = searcher.smart_search(
      query=description_from_new_boq,
      top_k=5,
      work_category=detected_category
  )
  # results contain rate, source_file, reranker_score

## Next: Notebook 05 — PostgreSQL Rate History
Store all 1429 items in SQL for:
  - "Average rate for brick work across all projects?"
  - "Rate trend: did prices go up project to project?"
  - Structured queries the vector search can't answer